In [34]:
import sys
print(sys.executable)


C:\Users\dell\anaconda3\python.exe


In [35]:
import skimage
print(skimage.__version__)


0.24.0


In [36]:
import os
print(os.getcwd())


C:\Users\dell\Cardialyse_Heart_Disease_Major Project\ECG\Diagnosis from ECG


In [37]:
OUTPUT_DIR = "final_model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [38]:
import os
import cv2
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
from scipy.signal import butter, filtfilt
import neurokit2 as nk
import joblib

In [39]:
MODEL_DIR = r"C:\Users\dell\Cardialyse_Heart_Disease_Major Project\ECG\Refined Modelling\final_model_output"

In [40]:
MODEL_DIR

'C:\\Users\\dell\\Cardialyse_Heart_Disease_Major Project\\ECG\\Refined Modelling\\final_model_output'

In [41]:
ensemble = joblib.load(os.path.join(MODEL_DIR, "hrv_ensemble_model.pkl"))
preprocessor = joblib.load(os.path.join(MODEL_DIR, "preprocessor.pkl"))
label_encoder = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))
valid_hrv_features = joblib.load(os.path.join(MODEL_DIR, "features.pkl"))

print("Models and artifacts loaded successfully.")

Models and artifacts loaded successfully.


In [42]:
def extract_hrv_from_ecg_image(
    img_path,
    pixels_per_mm=10,
    paper_speed=25,
    desired_fs=500
):
    """
    Converts ECG image → signal → HRV features (short-term only)
    """
    # Load image
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("Could not read ECG image.")

    # Convert to grayscale and denoise
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)

    # Binarize & skeletonize waveform
    thresh = threshold_otsu(gray)
    bw = gray < thresh
    skeleton = skeletonize(bw)

    xs, ys = [], []
    h, w = skeleton.shape
    for x in range(w):
        rows = np.where(skeleton[:, x])[0]
        if len(rows) > 0:
            xs.append(x)
            ys.append(int(np.median(rows)))

    xs = np.array(xs)
    ys = np.array(ys)

    if len(xs) < 100:
        raise ValueError("ECG signal too short or poorly extracted.")

    # Convert pixels → signal
    fs_est = pixels_per_mm * paper_speed
    t_pixels = xs / fs_est
    baseline = np.median(ys)
    mV_per_pixel = 1.0 / (10.0 * pixels_per_mm)
    signal = (baseline - ys) * mV_per_pixel

    # Resample to uniform time axis
    t_uniform = np.arange(t_pixels.min(), t_pixels.max(), 1.0 / desired_fs)
    signal_uniform = np.interp(t_uniform, t_pixels, signal)

    # Bandpass filter (0.5–40 Hz)
    def bandpass(sig, fs, low=0.5, high=40, order=3):
        nyq = 0.5 * fs
        b, a = butter(order, [low/nyq, high/nyq], btype="band")
        return filtfilt(b, a, sig)

    ecg_filtered = bandpass(signal_uniform, desired_fs)

    # R-peak detection
    ecg_cleaned = nk.ecg_clean(ecg_filtered, sampling_rate=desired_fs)
    _, info = nk.ecg_peaks(ecg_cleaned, sampling_rate=desired_fs)

    # HRV extraction
    hrv = nk.hrv(info, sampling_rate=desired_fs, show=False)

    # Keep only features used in training
    hrv = hrv.reindex(columns=valid_hrv_features)

    return hrv

In [43]:
def decision_with_confidence(probs, threshold=0.6):
    """
    Avoid forcing a disease when model confidence is low
    """
    max_prob = np.max(probs)
    pred_idx = np.argmax(probs)

    if max_prob < threshold:
        return "Uncertain – Needs further analysis", max_prob
    else:
        return label_encoder.inverse_transform([pred_idx])[0], max_prob


In [44]:
def mi_suspect_gate(hrv_row, probs):
    """
    Flags MI-suspect cases that HRV alone may label as Arrhythmia
    """
    mean_nn = hrv_row["HRV_MeanNN"]
    rmssd = hrv_row["HRV_RMSSD"]
    sdnn = hrv_row["HRV_SDNN"]

    mi_idx = label_encoder.transform(["MI"])[0]
    arr_idx = label_encoder.transform(["Arrhythmia"])[0]

    mi_prob = probs[mi_idx]
    arr_prob = probs[arr_idx]

    if (
        mi_prob > 0.10 and
        arr_prob > 0.50 and
        mean_nn < 800 and
        rmssd < 300 and
        sdnn < 300
    ):
        return "MI (HRV-based suspicion – morphology required)"

    return None


In [45]:
def diagnose_ecg_image(img_path):
    """
    Full pipeline: ECG image → HRV → disease prediction
    """
    print(f"\nProcessing ECG image: {img_path}")

    # Extract HRV features
    hrv_df = extract_hrv_from_ecg_image(img_path)

    # Preprocess
    hrv_processed = preprocessor.transform(hrv_df)

    # Predict
    probs = ensemble.predict_proba(hrv_processed)[0]

    # Confidence-based decision
    final_label, confidence = decision_with_confidence(probs)
    
    # MI suspicion override
    mi_override = mi_suspect_gate(hrv_df.iloc[0], probs)
    if mi_override:
        final_label = mi_override


    result = {
    "Predicted Disease": final_label,
    "Confidence": round(confidence, 3),
    "Class Probabilities": dict(
        zip(label_encoder.classes_, np.round(probs, 3))
    ),
    "Extracted HRV Features": hrv_df.iloc[0].to_dict()
}


    return result


In [46]:
if __name__ == "__main__":
    # Replace with path to NEW ECG image
    ECG_IMAGE_PATH = "ecg_sample.jpg"

    diagnosis = diagnose_ecg_image(ECG_IMAGE_PATH)

    print("\n=== DIAGNOSIS RESULT ===")
    print("Predicted Disease:", diagnosis["Predicted Disease"])
    print("\nProbabilities:")
    for k, v in diagnosis["Class Probabilities"].items():
        print(f"  {k}: {v}")


Processing ECG image: ecg_sample.jpg

=== DIAGNOSIS RESULT ===
Predicted Disease: Arrhythmia

Probabilities:
  Arrhythmia: 0.725
  CAD: 0.036
  MI: 0.115
  Normal: 0.124


In [47]:
if __name__ == "__main__":
    # Replace with path to NEW ECG image
    ECG_IMAGE_PATH = "ecg_sample_2.jpg"

    diagnosis = diagnose_ecg_image(ECG_IMAGE_PATH)

    print("\n=== DIAGNOSIS RESULT ===")
    print("Predicted Disease:", diagnosis["Predicted Disease"])
    print("\nProbabilities:")
    for k, v in diagnosis["Class Probabilities"].items():
        print(f"  {k}: {v}")


Processing ECG image: ecg_sample_2.jpg

=== DIAGNOSIS RESULT ===
Predicted Disease: Arrhythmia

Probabilities:
  Arrhythmia: 0.725
  CAD: 0.039
  MI: 0.113
  Normal: 0.122


In [48]:
if __name__ == "__main__":
    # Replace with path to NEW ECG image
    ECG_IMAGE_PATH = "ecg_sample_3.jpg"

    diagnosis = diagnose_ecg_image(ECG_IMAGE_PATH)

    print("\n=== DIAGNOSIS RESULT ===")
    print("Predicted Disease:", diagnosis["Predicted Disease"])
    print("\nProbabilities:")
    for k, v in diagnosis["Class Probabilities"].items():
        print(f"  {k}: {v}")


Processing ECG image: ecg_sample_3.jpg

=== DIAGNOSIS RESULT ===
Predicted Disease: Arrhythmia

Probabilities:
  Arrhythmia: 0.785
  CAD: 0.028
  MI: 0.09
  Normal: 0.098
